***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [5. 成像](5_0_introduction.ipynb)

***


导入标准模块:

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

NOTEBOOK_DIR = Path("5_Imaging") if Path("5_Imaging").exists() else Path(".")
NOTEBOOK_DIR = NOTEBOOK_DIR.resolve()
TOGGLE_PATH = NOTEBOOK_DIR.parent / "style" / "code_toggle.html"

In [ ]:
try:
    from IPython.display import HTML, Javascript
except ImportError:
    def HTML(*args, **kwargs):
        return None

    def Javascript(*args, **kwargs):
        return None

if TOGGLE_PATH.exists():
    HTML(TOGGLE_PATH.read_text(encoding="utf-8"))

***

## 5.A 综合成像中的匹配滤波器 <a id='imaging:sec:matched'></a>


匹配滤波器是一类用来在噪声背景中最大化目标信号信噪比的优化滤波器。它的基本思想，是用目标信号的模板与含噪输入做卷积，从而突出我们真正关心的结构。更多背景可参考 [Wikipedia: Matched Filter &#10548;](https://en.wikipedia.org/wiki/Matched_filter)。

你可能会问：既然已经知道感兴趣目标的大致形状，为什么还需要测量？在综合成像里，匹配滤波器依然有不少实际用途，例如：

1. 如果我们只关心某一特定空间尺度上的源，例如半解析星系，那么可以构造对应尺度的匹配滤波器来增强这些尺度上的信号，这通常也会带来更平滑、更对称的 PSF。
2. 如果我们研究的是某一类源，并掌握了它们的大致模板，那么即便模板并不完全精确，使用“近似匹配”的滤波器仍然常常能显著提高这一类源的信噪比。
3. 如果我们只关心致密源，例如点源，那么可以使用匹配滤波器压低大尺度结构。这在自校准中尤其有用，因为点源模型通常更容易、更快速地用于求解复增益（见 [8.3 节 &#10142;](../8_Calibration/8_3_2gc.ipynb)）。

虽然从数学上说任何模板都可以被拿来充当匹配滤波器，但在综合成像里，真正有意义的滤波器通常需要满足几个条件。首先，滤波器本身应当是空间上平滑的，例如高斯滤波器，因为天体源通常也具有一定平滑性。其次，滤波器的尺度存在下限，这一下限一般由 PSF 主瓣宽度决定；若滤波器比 PSF 主瓣还小，那么它通常不会对最终图像产生实际效果。


### 5.A.1 示例：高斯匹配滤波器

下面用一个简单例子说明匹配滤波器的作用。我们先生成一个单位强度的高斯源，再在图像中加入噪声，最后用与该源相匹配的滤波器尝试把它从噪声背景中恢复出来。下图展示的是未加噪声时的原始源。


In [ ]:
def generalGauss2d(x0, y0, sigmax, sigmay, amp=1.0, theta=0.0):
    """Return a normalized general 2-D Gaussian function.

    Parameters
    ----------
    x0, y0 : float
        Centre position.
    sigmax, sigmay : float
        Standard deviation along each principal axis.
    amp : float
        Peak amplitude.
    theta : float
        Rotation angle in degrees.
    """
    norm = amp
    rtheta = np.deg2rad(theta)

    a = (np.cos(rtheta) ** 2.0) / (2.0 * sigmax ** 2.0) + (np.sin(rtheta) ** 2.0) / (2.0 * sigmay ** 2.0)
    b = -(np.sin(2.0 * rtheta)) / (4.0 * sigmax ** 2.0) + (np.sin(2.0 * rtheta)) / (4.0 * sigmay ** 2.0)
    c = (np.sin(rtheta) ** 2.0) / (2.0 * sigmax ** 2.0) + (np.cos(rtheta) ** 2.0) / (2.0 * sigmay ** 2.0)
    return lambda x, y: norm * np.exp(-1.0 * (a * (x - x0) ** 2.0 - 2.0 * b * (x - x0) * (y - y0) + c * (y - y0) ** 2.0))



def apply_matched_filter(template, image):
    template = template / np.sqrt(np.sum(template ** 2))
    return np.fft.ifft2(np.fft.fft2(np.fft.fftshift(template)) * np.fft.fft2(image)).real



def estimate_peak_snr(image, mask):
    background = image[~mask]
    return float(np.max(image[mask]) / np.std(background))

In [ ]:
rng = np.random.default_rng(4)
imgSize = 512
sizeScale = 33.0
xpos, ypos = np.mgrid[0:imgSize, 0:imgSize].astype(float)

gFunc = generalGauss2d(300, 100, 7.0, 10.0, amp=1.0, theta=sizeScale)
trueSignal = gFunc(xpos, ypos)
single_source_mask = trueSignal > 0.25 * np.max(trueSignal)

fig = plt.figure(figsize=(9, 9))
plt.imshow(trueSignal, origin='lower')
plt.colorbar(shrink=0.75)
plt.title('Original source')

*图：二维高斯源。*


接下来向图像中加入零均值高斯噪声，使该源的峰值信噪比降到 0.3，这已经远低于一次观测中通常采用的检测阈值。匹配滤波器在这种背景下最容易体现“模板已知、噪声很强”时的优势。

In [ ]:
SNR = 0.3
noise_std = np.max(trueSignal) / SNR
noisySignal = trueSignal + noise_std * rng.standard_normal(trueSignal.shape)
print(f"Peak SNR before filtering = {estimate_peak_snr(noisySignal, single_source_mask):.3f}")

fig = plt.figure(figsize=(9, 9))
plt.imshow(noisySignal / np.max(np.abs(noisySignal)), origin='lower')
plt.colorbar(shrink=0.75)
plt.title('Original source with Gaussian noise added')

*图：信噪比为 0.3 的二维高斯源。*


既然我们知道原始源的形状和大小，就可以构造出相应的匹配滤波器。注意模板被放在滤波器图像的中央；由于后面执行的是卷积运算，因此我们并不需要事先知道原始源在整幅图中的具体位置。


In [ ]:
gFunc1 = generalGauss2d(imgSize / 2.0, imgSize / 2.0, 7.0, 10.0, amp=1.0, theta=33.0)
matchedFilter = gFunc1(xpos, ypos)
fig = plt.figure(figsize=(9, 9))
plt.imshow(matchedFilter, origin='lower')
plt.colorbar(shrink=0.75)
plt.title('Matched filter template')

*图：用于该源的匹配滤波器。*


将这个匹配滤波器应用到含噪图像上，本质上就是把模板与图像做卷积。正如前面介绍过的，这一步可以利用 [卷积定理 &#10142;](../2_Mathematical_Groundwork/2_7_fourier_theorems.ipynb) 高效完成。


In [ ]:
filteredSignal = apply_matched_filter(matchedFilter, noisySignal)
print(f"Peak SNR after matched filtering = {estimate_peak_snr(filteredSignal, single_source_mask):.3f}")

fig = plt.figure(figsize=(9, 9))
plt.imshow(filteredSignal / np.max(np.abs(filteredSignal)), origin='lower')
plt.colorbar(shrink=0.75)
plt.title('Matched filter applied')

*图：施加匹配滤波器后的结果图像。*


尽管我们人为加入了信噪比仅为 0.3 的强噪声，但在使用了合适的匹配滤波器之后，源信号依然明显高于背景起伏，因而能够被清楚地识别出来。


***

### 5.A.2 示例：不完全匹配的滤波器


在实际问题中，我们往往事先并不知道源的精确大小、形状或朝向。不过，即使模板并不完全正确，匹配滤波仍然常常能够给出“近似最优”的信噪比提升。


In [ ]:
gFunc1 = generalGauss2d(imgSize / 2.0, imgSize / 2.0, 2.0, 3.0, amp=1.0, theta=0.0)
matchedFilter = gFunc1(xpos, ypos)
fig = plt.figure(figsize=(8, 8))
plt.imshow(matchedFilter, origin='lower')
plt.title('Mismatched filter template')

In [ ]:
filteredSignal = apply_matched_filter(matchedFilter, noisySignal)
fig = plt.figure(figsize=(8, 8))
plt.imshow(filteredSignal, origin='lower')
plt.title('Mismatched filter applied')

***

### 5.A.3 多源情形


In [ ]:
gFunc = generalGauss2d(300, 100, 7.0, 10.0, amp=1.0, theta=33.0)
trueSignal = gFunc(xpos, ypos)

gFunc = generalGauss2d(400, 400, 20.0, 15.0, amp=0.3, theta=0.0)
trueSignal += gFunc(xpos, ypos)

gFunc = generalGauss2d(200, 300, 2.0, 3.0, amp=2.0, theta=45.0)
trueSignal += gFunc(xpos, ypos)

fig = plt.figure(figsize=(8, 8))
plt.imshow(trueSignal, origin='lower')
plt.title('Multi-source model image')

In [ ]:
SNR = 0.3
noise_std = np.max(trueSignal) / SNR
noisySignal = trueSignal + noise_std * rng.standard_normal(trueSignal.shape)
fig = plt.figure(figsize=(8, 8))
plt.imshow(noisySignal, origin='lower')
plt.title('Multi-source image with Gaussian noise')

In [ ]:
gFunc1 = generalGauss2d(imgSize / 2.0, imgSize / 2.0, 7.0, 10.0, amp=1.0, theta=33.0)
matchedFilter = gFunc1(xpos, ypos)
fig = plt.figure(figsize=(8, 8))
plt.imshow(matchedFilter, origin='lower')
plt.title('Filter tuned to source A')

In [ ]:
filteredSignal = apply_matched_filter(matchedFilter, noisySignal)
fig = plt.figure(figsize=(8, 8))
plt.imshow(filteredSignal, origin='lower')
plt.title('Matched-filter response for source A')

In [ ]:
gFunc1 = generalGauss2d(imgSize / 2.0, imgSize / 2.0, 20.0, 15.0, amp=0.3, theta=0.0)
matchedFilter = gFunc1(xpos, ypos)
fig = plt.figure(figsize=(8, 8))
plt.imshow(matchedFilter, origin='lower')
plt.title('Filter tuned to source B')

In [ ]:
filteredSignal = apply_matched_filter(matchedFilter, noisySignal)
fig = plt.figure(figsize=(8, 8))
plt.imshow(filteredSignal, origin='lower')
plt.title('Matched-filter response for source B')

In [ ]:
gFunc1 = generalGauss2d(imgSize / 2.0, imgSize / 2.0, 2.0, 3.0, amp=2.0, theta=45.0)
matchedFilter = gFunc1(xpos, ypos)
fig = plt.figure(figsize=(8, 8))
plt.imshow(matchedFilter, origin='lower')
plt.title('Filter tuned to source C')

In [ ]:
filteredSignal = apply_matched_filter(matchedFilter, noisySignal)
fig = plt.figure(figsize=(8, 8))
plt.imshow(filteredSignal, origin='lower')
plt.title('Matched-filter response for source C')

***